In [1]:
# Remove existing pyspark
!pip uninstall -y pyspark

# Install stable version
!pip install pyspark==3.5.1

Found existing installation: pyspark 3.5.1
Uninstalling pyspark-3.5.1:
  Successfully uninstalled pyspark-3.5.1
  Using cached pyspark-3.5.1-py2.py3-none-any.whl
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dataproc-spark-connect 1.0.2 requires pyspark[connect]~=4.0.0, but you have pyspark 3.5.1 which is incompatible.


In [2]:
!apt-get update
!apt-get install openjdk-11-jdk-headless -qq

import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"
os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]

!java -version

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Hit:2 https://cli.github.com/packages stable InRelease
Hit:3 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Fetched 3,632 B in 1s (3,300 B/s)
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
openjdk version "11.0.30" 2026-01-20
OpenJDK Runtime Environment (build 11.0.30+7-post-Ubuntu-1ubuntu122.04)
OpenJDK 64-Bit Server VM (build 11.0.3

In [3]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("CommonCrawlBigData") \
    .master("local[*]") \
    .config("spark.driver.memory", "6g") \
    .config("spark.executor.memory", "6g") \
    .config("spark.sql.shuffle.partitions", "200") \
    .getOrCreate()

spark

In [5]:
!wget https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-01.parquet

--2026-02-28 06:21:43--  https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-01.parquet
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 54.230.248.73, 54.230.248.205, 54.230.248.64, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|54.230.248.73|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 47673370 (45M) [application/x-www-form-urlencoded]
Saving to: ‘yellow_tripdata_2023-01.parquet’

yellow_tripdata_202 100%[===================>]  45.46M   176MB/s    in 0.3s    

2026-02-28 06:21:43 (176 MB/s) - ‘yellow_tripdata_2023-01.parquet’ saved [47673370/47673370]



In [6]:
df = spark.read.parquet("yellow_tripdata_2023-01.parquet")

print("Total records:", df.count())
df.printSchema()
df.show(5)

Total records: 3066766
root
 |-- VendorID: long (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: double (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: double (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)

+--------+--------------------+---------------------+---------------+-------------+----------+-------------

In [7]:
from pyspark.sql.functions import col

clean_df = df.filter(
    col("fare_amount").isNotNull() &
    col("trip_distance").isNotNull() &
    col("passenger_count").isNotNull()
)

print("After cleaning:", clean_df.count())

After cleaning: 2995023


In [8]:
from pyspark.sql.functions import when

labeled_df = clean_df.withColumn(
    "high_fare",
    when(col("fare_amount") > 30, 1).otherwise(0)
)

In [9]:
from pyspark.ml.feature import VectorAssembler

feature_cols = [
    "trip_distance",
    "passenger_count",
    "PULocationID",
    "DOLocationID",
    "payment_type"
]

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features"
)

final_df = assembler.transform(labeled_df)
final_df.select("features", "high_fare").show(5)

+--------------------+---------+
|            features|high_fare|
+--------------------+---------+
|[0.97,1.0,161.0,1...|        0|
|[1.1,1.0,43.0,237...|        0|
|[2.51,1.0,48.0,23...|        0|
|[1.9,0.0,138.0,7....|        0|
|[1.43,1.0,107.0,7...|        0|
+--------------------+---------+
only showing top 5 rows



In [10]:
train, test = final_df.randomSplit([0.8, 0.2], seed=42)

In [11]:
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator

lr = LogisticRegression(
    labelCol="high_fare",
    featuresCol="features",
    maxIter=20
)

lr_model = lr.fit(train)
lr_pred = lr_model.transform(test)

evaluator = BinaryClassificationEvaluator(labelCol="high_fare")
lr_auc = evaluator.evaluate(lr_pred)

print("Logistic Regression AUC:", lr_auc)

Logistic Regression AUC: 0.9622610215249383


In [12]:
from pyspark.ml.classification import DecisionTreeClassifier

dt = DecisionTreeClassifier(
    labelCol="high_fare",
    featuresCol="features",
    maxDepth=10
)

dt_model = dt.fit(train)
dt_pred = dt_model.transform(test)

dt_auc = evaluator.evaluate(dt_pred)

print("Decision Tree AUC:", dt_auc)

Decision Tree AUC: 0.9107003941678933


In [13]:
from pyspark.ml.classification import RandomForestClassifier

rf = RandomForestClassifier(
    labelCol="high_fare",
    featuresCol="features",
    numTrees=50,
    maxDepth=10
)

rf_model = rf.fit(train)
rf_pred = rf_model.transform(test)

rf_auc = evaluator.evaluate(rf_pred)

print("Random Forest AUC:", rf_auc)

Random Forest AUC: 0.991672361874942


In [14]:
from pyspark.ml.classification import GBTClassifier

gbt = GBTClassifier(
    labelCol="high_fare",
    featuresCol="features",
    maxIter=20,
    maxDepth=5
)

gbt_model = gbt.fit(train)
gbt_pred = gbt_model.transform(test)

gbt_auc = evaluator.evaluate(gbt_pred)

print("Gradient Boosted Trees AUC:", gbt_auc)

Gradient Boosted Trees AUC: 0.9925102485535814


In [15]:
results = {
    "Logistic Regression": lr_auc,
    "Decision Tree": dt_auc,
    "Random Forest": rf_auc,
    "Gradient Boosted Trees": gbt_auc
}

for model, score in results.items():
    print(f"{model}: {score:.4f}")

Logistic Regression: 0.9623
Decision Tree: 0.9107
Random Forest: 0.9917
Gradient Boosted Trees: 0.9925


In [16]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

accuracy_eval = MulticlassClassificationEvaluator(
    labelCol="high_fare",
    predictionCol="prediction",
    metricName="accuracy"
)

f1_eval = MulticlassClassificationEvaluator(
    labelCol="high_fare",
    predictionCol="prediction",
    metricName="f1"
)

print("RF Accuracy:", accuracy_eval.evaluate(rf_pred))
print("RF F1:", f1_eval.evaluate(rf_pred))

RF Accuracy: 0.9772525149377878
RF F1: 0.97684047696805


In [17]:
from pyspark.sql.functions import col, count, avg

# Null count per column
null_counts = df.select([
    count(col(c)).alias(c) for c in df.columns
])

# Basic statistics
stats_df = df.select(
    avg("fare_amount").alias("avg_fare"),
    avg("trip_distance").alias("avg_distance")
)

# Payment type distribution
payment_dist = df.groupBy("payment_type").count()

# Save as CSV for Tableau
null_counts.coalesce(1).write.mode("overwrite").csv("dashboard1_nulls", header=True)
stats_df.coalesce(1).write.mode("overwrite").csv("dashboard1_stats", header=True)
payment_dist.coalesce(1).write.mode("overwrite").csv("dashboard1_payment", header=True)

In [18]:
import pandas as pd

performance_data = pd.DataFrame({
    "Model": ["Logistic Regression", "Decision Tree", "Random Forest", "Gradient Boosted Trees"],
    "AUC": [lr_auc, dt_auc, rf_auc, gbt_auc]
})

performance_data.to_csv("dashboard2_model_performance.csv", index=False)

In [19]:
from pyspark.sql.functions import hour, sum as spark_sum

# Extract hour
hourly_df = df.withColumn("pickup_hour", hour("tpep_pickup_datetime"))

hourly_revenue = hourly_df.groupBy("pickup_hour") \
    .agg(
        spark_sum("fare_amount").alias("total_revenue")
    )

passenger_fare = df.groupBy("passenger_count") \
    .agg(
        avg("fare_amount").alias("avg_fare")
    )

location_revenue = df.groupBy("PULocationID") \
    .agg(
        spark_sum("fare_amount").alias("location_revenue")
    )

# Save
hourly_revenue.coalesce(1).write.mode("overwrite").csv("dashboard3_hourly", header=True)
passenger_fare.coalesce(1).write.mode("overwrite").csv("dashboard3_passenger", header=True)
location_revenue.coalesce(1).write.mode("overwrite").csv("dashboard3_location", header=True)

In [20]:
import time

scaling_results = []

for partitions in [50, 100, 200, 400]:
    temp_df = df.repartition(partitions)

    start = time.time()
    temp_df.groupBy("payment_type").count().collect()
    end = time.time()

    scaling_results.append({
        "Partitions": partitions,
        "Execution_Time_Seconds": end - start
    })

scaling_df = pd.DataFrame(scaling_results)
scaling_df.to_csv("dashboard4_scalability.csv", index=False)

In [21]:
!zip -r dashboard_files.zip dashboard*

  adding: dashboard1_nulls/ (stored 0%)
  adding: dashboard1_nulls/._SUCCESS.crc (stored 0%)
  adding: dashboard1_nulls/_SUCCESS (stored 0%)
  adding: dashboard1_nulls/.part-00000-7419840d-8a19-4749-af86-bd678c6e1a52-c000.csv.crc (stored 0%)
  adding: dashboard1_nulls/part-00000-7419840d-8a19-4749-af86-bd678c6e1a52-c000.csv (deflated 52%)
  adding: dashboard1_payment/ (stored 0%)
  adding: dashboard1_payment/part-00000-d928df2e-d564-4e4c-8626-b37a83a86bdc-c000.csv (deflated 2%)
  adding: dashboard1_payment/._SUCCESS.crc (stored 0%)
  adding: dashboard1_payment/.part-00000-d928df2e-d564-4e4c-8626-b37a83a86bdc-c000.csv.crc (stored 0%)
  adding: dashboard1_payment/_SUCCESS (stored 0%)
  adding: dashboard1_stats/ (stored 0%)
  adding: dashboard1_stats/._SUCCESS.crc (stored 0%)
  adding: dashboard1_stats/.part-00000-5c7146b7-b3ea-4ce5-8b56-bacab6afd179-c000.csv.crc (stored 0%)
  adding: dashboard1_stats/_SUCCESS (stored 0%)
  adding: dashboard1_stats/part-00000-5c7146b7-b3ea-4ce5-8b56-bacab